In [ ]:
!git clone 'https://github.com/labwons/pylabwons-archive.git'
%cd pylabwons-archive

!pip install -e .
%cd ..

In [ ]:
import requests
from pykrx import stock
from pykrx.website.comm import webio

_session = requests.Session()

def login_krx(login_id: str, login_pw: str) -> bool:
    """
    KRX data.krx.co.kr 로그인 후 세션 쿠키(JSESSIONID)를 갱신합니다.

    로그인 흐름:
    1. GET MDCCOMS001.cmd  → 초기 JSESSIONID 발급
    2. GET login.jsp       → iframe 세션 초기화
    3. POST MDCCOMS001D1.cmd → 실제 로그인
    4. CD011(중복 로그인) → skipDup=Y 추가 후 재전송
    """
    _LOGIN_PAGE = "https://data.krx.co.kr/contents/MDC/COMS/client/MDCCOMS001.cmd"
    _LOGIN_JSP  = "https://data.krx.co.kr/contents/MDC/COMS/client/view/login.jsp?site=mdc"
    _LOGIN_URL  = "https://data.krx.co.kr/contents/MDC/COMS/client/MDCCOMS001D1.cmd"
    _UA = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
    )

    # 초기 세션 발급
    _session.get(_LOGIN_PAGE, headers={"User-Agent": _UA}, timeout=15)
    _session.get(_LOGIN_JSP, headers={"User-Agent": _UA, "Referer": _LOGIN_PAGE}, timeout=15)

    payload = {
        "mbrNm": "", "telNo": "", "di": "", "certType": "",
        "mbrId": login_id, "pw": login_pw,
    }
    headers = {"User-Agent": _UA, "Referer": _LOGIN_PAGE}

    # 로그인 POST
    resp = _session.post(_LOGIN_URL, data=payload, headers=headers, timeout=15)
    data = resp.json()
    error_code = data.get("_error_code", "")

    # CD011 중복 로그인 처리
    if error_code == "CD011":
        payload["skipDup"] = "Y"
        resp = _session.post(_LOGIN_URL, data=payload, headers=headers, timeout=15)
        data = resp.json()
        error_code = data.get("_error_code", "")

    return error_code == "CD001"  # CD001 = 정상


def _session_post_read(self, **params):
    return _session.post(self.url, headers=self.headers, data=params)

def _session_get_read(self, **params):
    return _session.get(self.url, headers=self.headers, params=params)

webio.Post.read = _session_post_read
webio.Get.read = _session_get_read
login_krx('jhlee0319', 'dwg!4r4r6y1q')

# Build

In [17]:
import pylabwons as lw
import pylabwons_stub as lws


baseline = lws.Baseline(logger=lw.Logger())
baseline.build()

2026-03-03 17:35:23 [BUILD BASELINE] @20260303
2026-03-03 17:35:23 FETCH AFTER MARKET DATA ON 20260303
2026-03-03 17:35:25 FETCH [GENERAL INFO] ... OK
2026-03-03 17:35:26 FETCH [MARKET CAP] ... OK
2026-03-03 17:35:27 FETCH [FOREIGN RATE] ... OK
2026-03-03 17:35:28 FETCH [MARKET CAP TYPE] ... OK
2026-03-03 17:36:36 FETCH [RETURNS] ... OK
2026-03-03 17:36:36 .............................. 2657 STOCKS / RUNTIME: 73.16s
2026-03-03 17:36:36 Unable to cast <yoyProfit: object -> numeric>, could not convert string to float: '적자전환'
2026-03-03 17:36:36 Unable to cast <estimatedProfitRateGrowth: object -> numeric>, could not convert string to float: '흑자전환'
2026-03-03 17:36:36 Unable to cast <fiscalNetProfitGrowth: object -> numeric>, could not convert string to float: '흑자전환'
2026-03-03 17:36:36 Unable to cast <yoyProfitRate: object -> numeric>, could not convert string to float: '적자전환'
2026-03-03 17:36:36 Unable to cast <yoyNetProfit: object -> numeric>, could not convert string to float: '적자지속'


In [19]:
baseline.log.save()
baseline.date

{'baseline': {'date': '20260303'},
 'aftermarket': {'date': ''},
 'numbers': {'date': ''},
 'wics': {'date': ''}}

In [6]:
if lws.HOST == 'google_colab':
    from google.colab import userdata
    import os
    if not os.getcwd().endswith('pylabwons-archive'):
        %cd pylabwons-archive

    GIT_TOKEN = userdata.get('pylabwons-archive-github')
    GIT_USER = "labwons"
    GIT_REPO = "pylabwons-archive"
    GIT_MAIL = "snob.labwons@gmail.com"

    !git config --global user.name "{GIT_USER}"
    !git config --global user.email "{GIT_MAIL}"
    !git remote set-url origin "https://{GIT_USER}:{GIT_TOKEN}@://github.com{GIT_USER}/{GIT_REPO}.git"
    # !git add .
    # !git commit -m "COMMIT AND PUSH FROM COLAB"
    !git push origin main

fatal: unable to access 'https://://github.comlabwons/pylabwons-archive.git/': URL using bad/illegal format or missing URL
